# AI interview analysis workflow

A sanitized source excerpt from Sally You's Colab project. The original project processed 29 interview transcripts; the original transcripts and outputs are not distributed. This notebook preserves the original draft-codebook and two-pass feature-synthesis logic, with credentials and identifying examples removed.

**Run status:** local helper checks only; live Gemini results have not been revalidated. The original `google-generativeai` SDK is legacy, and the historical model names may need updating before live use. Model prompts requesting verbatim evidence or rigor do not guarantee either. Review outputs against the source transcripts.

Run the setup cells first, then choose the draft-codebook action, feature-synthesis action, or both. Each action sends transcript text to Gemini and may incur API charges. Use only data you are permitted to send. See the README for provenance and limitations.


## 1. Install the original project dependencies

This retains the historical SDK rather than presenting a later rewrite as the original work. The unrelated package-uninstall command from the export has been omitted.


In [ ]:
%pip -q install google-generativeai python-docx pymupdf


## 2. Choose an input folder

Copy `examples/interview_01.txt` and `examples/interview_02.txt` into the Google Drive folder below to try synthetic inputs. Both are fictional. The folder should contain input transcripts only; keep generated reports elsewhere.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Put the synthetic example files, or your own approved transcripts, in this folder.
FOLDER = "/content/drive/MyDrive/interview-transcripts"


## 3. Configure the model and load the shared helpers

Supply your own key through an environment variable or the hidden prompt. No credential is saved in this notebook. API access is required for the later generation cells, not for reading the source.


In [ ]:
# ==== MODEL SETUP ====
import os, re, json, time, fitz, google.generativeai as genai
from getpass import getpass
from datetime import datetime
from docx import Document
from docx.shared import Pt
from docx.oxml.ns import qn

API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or getpass("Paste your Gemini API key (hidden): ")
genai.configure(api_key=API_KEY)

MODEL_PRO  = "gemini-2.5-pro"
MODEL_NAME = MODEL_PRO

# ==== FILE READERS ====
def read_txt(p):
    with open(p, "r", encoding="utf-8", errors="ignore") as f: return f.read()
def read_docx(p):
    from docx import Document
    return "\n".join(para.text for para in Document(p).paragraphs)
def read_pdf(p):
    text = []
    with fitz.open(p) as doc:
        for page in doc: text.append(page.get_text("text"))
    return "\n".join(text)
def read_any(p):
    pl = p.lower()
    if pl.endswith(".txt"):  return read_txt(p)
    if pl.endswith(".docx"): return read_docx(p)
    if pl.endswith(".pdf"):  return read_pdf(p)
    return ""

def list_transcripts(folder):
    files = sorted([os.path.join(folder,f) for f in os.listdir(folder)
                    if f.lower().endswith((".txt",".docx",".pdf"))])
    texts = [(os.path.basename(p), read_any(p)) for p in files]
    texts = [(n, t.strip()) for n,t in texts if t and t.strip()]
    return texts

texts = list_transcripts(FOLDER)
all_filenames = [n for n,_ in texts]
print(f"✅ Found {len(texts)} transcripts.")

# ==== DOCX EXPORTER ====
def save_docx(markdownish_text, out_path):
    doc = Document()
    style = doc.styles['Normal']
    style.font.name = 'Calibri'
    style._element.rPr.rFonts.set(qn('w:eastAsia'), 'Calibri')
    style.font.size = Pt(11)
    for line in markdownish_text.splitlines():
        s = line.strip()
        if not s: doc.add_paragraph(""); continue
        if s.startswith("### "): doc.add_heading(s[4:], level=3)
        elif s.startswith("## "): doc.add_heading(s[3:], level=2)
        elif s.startswith("# "):  doc.add_heading(s[2:], level=1)
        elif s.lower().startswith("code:"):
            p = doc.add_paragraph(); p.add_run(s).bold = True
        elif re.match(r"^(definition|when to use|when not to use|evidence|analyst memo):", s, flags=re.I):
            label, content = s.split(":", 1)
            p = doc.add_paragraph(); p.add_run(label + ": ").bold = True; p.add_run(content.strip())
        elif s.startswith(("- ", "* ")):
            doc.add_paragraph(s[2:], style="List Bullet")
        else:
            doc.add_paragraph(s)
    doc.save(out_path)
    return out_path

# ==== STATE ====
STATE_PATH = os.path.join(FOLDER, "_analysis_state.json")
def load_state():
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH, "r", encoding="utf-8") as f: return json.load(f)
    return {"processed_files": [], "codebook_v1_path": None, "last_run": None}
def save_state(state):
    state["last_run"] = datetime.now().isoformat(timespec="seconds")
    with open(STATE_PATH, "w", encoding="utf-8") as f: json.dump(state, f, indent=2)
    return STATE_PATH
state = load_state()



## 4. Draft a codebook (optional action)

The following prompt produces an initial draft for review. Its methodological instructions describe the intended output; they are not evidence of validated qualitative analysis.


In [ ]:
THEMATIC_ANALYSIS_PROMPT_REFINED = """You are a qualitative researcher with 20 years of experience conducting thematic analysis following Braun & Clarke (2006, 2019).
You have just read a set of interview transcripts. Your goal is to generate an *initial thematic codebook* that demonstrates academic and analytic rigor.

Follow these detailed guidelines when producing the codebook:

---

### 1. Analytic Framing
- Clearly state the **analytic orientation** (semantic, latent, or hybrid) and your **epistemological stance** (e.g., realist, constructivist, or contextualist).
- Briefly summarize the *purpose of analysis* and *how the themes answer the research question*.

---

### 2. Codebook Structure
Organize the output into **Themes**, each with **Subthemes** and **Associated Codes**.

For each **Theme**, include:
- **Theme Name:** a concise, conceptually rich title
- **Theme Summary:** 3–5 sentences describing the central organizing concept and how it relates to the research goal
- **Subthemes (if applicable)**
- **Analyst Memo:** a short reflective note interpreting what this theme reveals about participants’ perspectives, tensions, or contradictions

Then, for each **Code**, list details in this *line-by-line format* (not a table):

**Code:** <short label>
**Definition:** <one-sentence conceptual definition>
**When to Use:** <criteria for application>
**When Not to Use:** <common near-misses to exclude>
**Evidence (5 excerpts):**
- [Transcript: <ID or filename>] "<verbatim quote #1>"
- [Transcript: <ID or filename>] "<verbatim quote #2>"
- [Transcript: <ID or filename>] "<verbatim quote #3>"
- [Transcript: <ID or filename>] "<verbatim quote #4>"
- [Transcript: <ID or filename>] "<verbatim quote #5>"

Formatting rules for evidence:
- Extract five direct, unedited quotes from the transcripts that exemplify this code.
- Each excerpt should be copied exactly as written, preserving punctuation, grammar, and filler words (e.g., "um","you know",etc.)
- Do not summarize or paraphrase any quotes
- Distribute excerpts across different transcripts when possible; avoid duplicates from the same passage.
- If fewer than 5 relevant excerpts exist, return all available and add: **Note:** fewer than 5 excerpts found.
- If your environment provides transcript IDs or filenames, **always** tag each excerpt with that ID/filename exactly as given.

Leave a blank line between codes for readability.

---

### 3. Thematic Rigor
- Ensure each theme is **mutually exclusive** and has a **distinct central organizing concept**.
- Avoid overlapping or redundant codes; collapse similar ones when necessary.
- Show **hierarchical relationships** (themes ↔ subthemes ↔ codes) clearly.

---

### 4. Cross-Cutting Dimensions
Include a short section listing **cross-cutting dimensions** (concepts that appear across multiple themes), such as *trust*, *efficiency*, or *ethical responsibility*.

---

### 5. Reflexive Notes
At the end, add a short **Analyst Reflection** summarizing:
- How your interpretive stance influenced theme development
- Which areas require further refinement or team review in the next coding cycle

---

### Output Formatting
- Use clear markdown or Word-friendly structure:
  - `## Theme:` headings
  - `### Subtheme:` subheadings
  - Line-by-line formatting for each code as shown above
- Avoid tables entirely — make the output readable and ready for `.docx` export.

---

The identifying example has been removed from this portfolio copy. Follow the output format above.

Generate your full codebook following this structure and formatting."""


In [ ]:
# 1) Build roster + labeled corpus
roster = "Transcript roster:\n" + "\n".join(f"- {n}" for n,_ in texts)
corpus = "\n\n".join(
    f"=== BEGIN TRANSCRIPT {i}: {n} ===\n{t}\n=== END TRANSCRIPT {i} ==="
    for i,(n,t) in enumerate(texts, 1)
)
prompt = (
    THEMATIC_ANALYSIS_PROMPT_REFINED
    + "\n\nIMPORTANT: When listing Evidence (5 verbatim excerpts) for each code, "
      "distribute across different transcripts when possible and tag quotes exactly as [Transcript: <filename>].\n\n"
    + roster + "\n\n" + corpus
)

model = genai.GenerativeModel(MODEL_PRO)
resp = model.generate_content(prompt, request_options={"timeout": 900})
codebook_v1 = resp.text

from google.colab import files
ts = datetime.now().strftime("%Y-%m-%d_%H%M")
# Auto-download DOCX
out_path = f"/content/Codebook_v1_{ts}.docx"
# Save DOCX in the folder
# out_path = os.path.join(FOLDER, f"Codebook_v1_{ts}.docx")
save_docx(codebook_v1, out_path)

# trigger immediate download
try:
    files.download(out_path)
    print("⬇️ Auto-download started for:", out_path)
except Exception as e:
    print("⚠️ Auto-download failed:", e)
    print("You can still download it manually from the left Files panel.")

# 3) Update state
state["processed_files"] = all_filenames
state["codebook_v1_path"] = out_path
save_state(state)
print("✅ State updated at:", STATE_PATH)


## 5. Extract and consolidate feature requests (optional action)

Pass 1 requests JSON feature objects for each transcript. Pass 2 asks Gemini to merge overlapping requests into a Word report. JSON parsing is basic, and semantic merging and quotation fidelity still require human review.


In [ ]:
import time, textwrap, json
import google.generativeai as genai
from datetime import datetime
from google.colab import files as colab_files

# --- 0. Sanity checks ---
if "texts" not in globals():
    raise RuntimeError(
        "❌ 'texts' is not defined. Run your transcript-loading cell first "
        "(the one that builds `texts = [(filename, content), ...]`)."
    )

try:
    MODEL_NAME
except NameError:
    MODEL_NAME = "gemini-2.5-pro"

model = genai.GenerativeModel(MODEL_NAME)

# --- 1. Helper: robust model call with retries ---
def gen_with_retry(prompt, tries=4, timeout_sec=600):
    last_err = None
    for k in range(tries):
        try:
            resp = model.generate_content(
                prompt,
                request_options={"timeout": timeout_sec}
            )
            return resp.text
        except Exception as e:
            last_err = e
            print(f"⚠️ Attempt {k+1}/{tries} failed: {type(e).__name__} → {e}")
            if k < tries - 1:
                wait = 3 + 3*k
                print(f"⏳ Waiting {wait}s before retry...")
                time.sleep(wait)
    raise last_err

# --- 2. Prompt to extract features from ONE transcript as JSON ---
def make_feature_json_prompt(filename, content):
    return textwrap.dedent(f"""
    You are analyzing a single customer / user interview transcript about an analytics / AI platform
    for investment professionals.

    Your task is to extract **all requested or implied product features** the participant wants the
    platform to have.

    A "feature" can be:
    - A specific capability (e.g., Excel plugin, backtesting, explainable scores)
    - A workflow improvement (e.g., auto-ingest earnings transcripts, one-click thesis summaries)
    - A UX / integration requirement (e.g., integrates with Bloomberg, low-friction UI)
    - A governance / data feature (e.g., SOC2, audit trail, configurable access control)

    ### Transcript
    [Transcript: {filename}]
    \"\"\"{content}\"\"\"

    ------------------------------------------------
    ### Your Output

    Return ONLY **valid JSON** and nothing else.
    The JSON must be a list of feature objects with this exact schema:

    [
      {{
        "transcript": "{filename}",
        "feature_label": "short label, 3–8 words",
        "description": "2–3 sentences describing what this feature does from the user's perspective.",
        "pain_point": "1–3 sentences on the workflow problem or job-to-be-done this feature solves.",
        "evidence": [
          "verbatim quote 1 from transcript that supports this feature",
          "verbatim quote 2 (if available)",
          "... (more quotes if clearly distinct)"
        ]
      }},
      ...
    ]

    Rules:
    - If the participant does NOT clearly request or imply any features, return an empty list: []
    - Quotes in "evidence" must be **verbatim** (no paraphrasing).
    - Do NOT include markdown, comments, or text outside the JSON.
    """).strip()

# Small helper to make JSON parsing more robust
def parse_json_safely(raw_text):
    """
    Try to parse JSON, even if the model accidentally wraps it in code fences.
    """
    text = raw_text.strip()
    # Strip ```json ... ``` if present
    if text.startswith("```"):
        text = text.strip("`")
        # remove possible leading 'json' etc.
        first_newline = text.find("\n")
        if first_newline != -1:
            text = text[first_newline+1:].strip()
    return json.loads(text)

# --- 3. Pass 1: extract feature objects per transcript ---
all_feature_items = []  # list of dicts

for filename, content in texts:
    print(f"🔎 Extracting feature JSON from: {filename}")
    prompt = make_feature_json_prompt(filename, content)
    raw = gen_with_retry(prompt)

    try:
        data = parse_json_safely(raw)
        if not isinstance(data, list):
            print(f"⚠️ Unexpected JSON root type in {filename}; expected list, got {type(data)}. Skipping.")
            continue
        for obj in data:
            # basic sanity check
            if not isinstance(obj, dict):
                continue
            obj.setdefault("transcript", filename)
            obj.setdefault("feature_label", "")
            obj.setdefault("description", "")
            obj.setdefault("pain_point", "")
            obj.setdefault("evidence", [])
            all_feature_items.append(obj)
    except Exception as e:
        print(f"❌ Failed to parse JSON for {filename}: {e}")
        # Optionally keep raw text for manual inspection later
        continue

print(f"\n✅ Collected {len(all_feature_items)} feature instances across {len(texts)} transcripts.")

if not all_feature_items:
    raise SystemExit("No feature data extracted; nothing to merge.")

# --- 4. Build a plain-text list of all feature mentions for merging ---
# This keeps everything human-readable for the model in the next pass.
feature_blocks = []
for idx, feat in enumerate(all_feature_items, start=1):
    transcript = feat.get("transcript", "UNKNOWN")
    label      = feat.get("feature_label", "").strip()
    desc       = feat.get("description", "").strip()
    pain       = feat.get("pain_point", "").strip()
    quotes     = feat.get("evidence", []) or []

    quotes_block = "\n".join(
        f'- [Transcript: {transcript}] "{q.strip()}"'
        for q in quotes if q and isinstance(q, str)
    ) or "- [NOTE] No quotes captured (empty evidence list)."

    block = textwrap.dedent(f"""
    ### Feature Instance {idx}
    **Transcript:** {transcript}
    **Proposed Label:** {label}
    **Description:** {desc}
    **Pain Point / Job-to-be-done:** {pain}
    **Evidence:**
    {quotes_block}
    """).strip()

    feature_blocks.append(block)

merged_input_blob = "\n\n".join(feature_blocks)

# --- 5. Pass 2: Ask Gemini to MERGE overlapping features ---
merge_prompt = textwrap.dedent(f"""
You are synthesizing product feature requests across many customer interviews.

Below is a list of **feature instances** extracted transcript by transcript.
Some are overlapping, redundant, or phrased slightly differently but represent the same underlying feature.

Your tasks:

1. **Cluster and merge overlapping features.**
   - If two or more feature instances describe essentially the same capability or requirement, merge them.
   - Keep distinct features separate; do NOT over-merge unrelated ideas.

2. For each **final merged feature**, output in this structure:

## Feature: <concise merged label, 3–8 words>
**Problem / Job-to-be-done:** <2–4 sentences on the user problem this feature solves.>
**What users are asking for:** <2–5 sentences summarizing what the feature needs to do (functionally and experientially).>
**Supporting Evidence (verbatim quotes):**
- [Transcript: <filename>] "<quote 1>"
- [Transcript: <filename>] "<quote 2>"
- ...

Guidelines:
- Use quotes **exactly** as given (verbatim; no paraphrasing).
- When multiple transcripts mention the same feature, include representative quotes from **different** transcripts where available.
- If a feature appears only once in one transcript, still include it as its own feature if it is conceptually meaningful.
- Do NOT invent new features that are not grounded in the input.
- Do NOT include the raw feature instance list in the output; only the merged, cleaned feature list in the format above.

------------------------------------------------
### Raw Feature Instances (to be merged)
{merged_input_blob}
""").strip()

print("\n🧠 Merging overlapping features across transcripts...")
merged_features_text = gen_with_retry(merge_prompt, timeout_sec=900)

# --- 6. Wrap in a top-level heading & save to DOCX ---
final_text = "# Exhaustive Merged Feature List Across Transcripts\n\n" + merged_features_text

ts = datetime.now().strftime("%Y-%m-%d_%H%M")
out_path = f"/content/Merged_Feature_List_{ts}.docx"
save_docx(final_text, out_path)

try:
    colab_files.download(out_path)
    print("⬇️ Auto-download started:", out_path)
except Exception as e:
    print("✅ Saved merged feature report at:", out_path)
    print("   Auto-download failed with:", repr(e))
